# 5. Inference — поиск и submission
Гибридный поиск: FAISS (dense) + BM25 (sparse) → RRF fusion → топ-5 → submission.csv

In [ ]:
!pip install faiss-cpu rank-bm25 -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import pickle
import numpy as np
import faiss
import pandas as pd
import torch
from pathlib import Path
from collections import defaultdict
from transformers import AutoModel, AutoTokenizer
from rank_bm25 import BM25Okapi
from tqdm import tqdm

In [ ]:
# ── Конфиг ──────────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
E5_MODEL = 'intfloat/multilingual-e5-large'  # лучше чем e5-small-en-ru

HOME_DIR   = Path('/kaggle/input/competitions/multi-lingual-video-fragment-retrieval-challenge/video-rag')
TEST_CSV   = HOME_DIR / 'test/test.csv'

# Транскрипции: используем Large-v3 если есть, иначе бейзлайн
TRANSCRIPTS_LARGE = Path('transcripts_large.pkl')
TRANSCRIPTS_PKL   = HOME_DIR / 'transcripts.pkl'
TRANSCRIPTS_PATH  = TRANSCRIPTS_LARGE if TRANSCRIPTS_LARGE.exists() else TRANSCRIPTS_PKL

# Параметры поиска
K_DENSE   = 120   # кандидаты из FAISS
K_SPARSE  = 100   # кандидаты из BM25
RRF_K     = 60    # параметр RRF
TOP_N     = 5     # финальный топ

print(f'Device: {DEVICE}')
print(f'Транскрипции: {TRANSCRIPTS_PATH}')

## 1. Загрузка транскрипций и построение чанков

In [ ]:
with open(TRANSCRIPTS_PATH, 'rb') as f:
    transcripts = pickle.load(f)

print(f'Загружено видео: {len(transcripts)}')
# Показываем пример ключа
sample_key = list(transcripts.keys())[0]
print(f'Пример ключа: {sample_key}')
print(f'Первые 2 сегмента: {transcripts[sample_key][:2]}')

In [ ]:
# Строим чанки из отдельных Whisper-сегментов (НЕ 30-секундные окна!)
# Каждый сегмент = один чанк → IoU будет точным (avg gt = 3 сек)
# Добавляем контекст: предыдущий + следующий сегмент в текст эмбеддинга

chunks = []  # {video_file, start, end, text, embed_text}

for video_key, segments in transcripts.items():
    if not segments:
        continue

    # Нормализуем video_file → только stem без расширения
    # Ключи могут быть: 'videos/video_abc123.opus' или 'videos/video_abc123'
    video_stem = Path(video_key).stem  # 'video_abc123'

    for i, seg in enumerate(segments):
        if not seg.get('text', '').strip():
            continue

        # Контекст для эмбеддинга: prev + current + next
        context_parts = []
        if i > 0 and segments[i-1].get('text', '').strip():
            context_parts.append(segments[i-1]['text'].strip())
        context_parts.append(seg['text'].strip())
        if i < len(segments)-1 and segments[i+1].get('text', '').strip():
            context_parts.append(segments[i+1]['text'].strip())

        chunks.append({
            'video_file': video_stem,
            'start': float(seg['start']),
            'end': float(seg['end']),
            'text': seg['text'].strip(),
            'embed_text': ' '.join(context_parts),  # эмбеддим с контекстом
        })

print(f'Всего чанков: {len(chunks)}')
print(f'Пример чанка: {chunks[0]}')

## 2. E5 эмбеддинги + FAISS индекс

In [ ]:
print(f'Загружаем {E5_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(E5_MODEL)
e5_model  = AutoModel.from_pretrained(E5_MODEL).to(DEVICE).eval()
print('Готово')

def embed_texts(texts, prefix='passage: ', batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = [prefix + t for t in texts[i:i+batch_size]]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=512, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out = e5_model(**inputs)
            emb = out.last_hidden_state[:, 0]
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            all_embeddings.append(emb.cpu().numpy())
    return np.vstack(all_embeddings)

In [ ]:
print('Вычисляем эмбеддинги чанков...')
chunk_texts = [c['embed_text'] for c in chunks]
chunk_embeddings = embed_texts(chunk_texts, prefix='passage: ', batch_size=32)
print(f'Shape: {chunk_embeddings.shape}')

In [ ]:
dim   = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f'FAISS индекс: {index.ntotal} векторов, dim={dim}')

## 3. BM25 индекс

In [ ]:
# Токенизация: просто split (можно улучшить через nltk/spacy)
corpus = [c['text'].lower().split() for c in chunks]
bm25   = BM25Okapi(corpus)
print(f'BM25 индекс построен: {len(corpus)} документов')

## 4. Поиск: RRF fusion

In [ ]:
def rrf_fusion(dense_indices, sparse_indices, k=60):
    """Reciprocal Rank Fusion. Возвращает {chunk_idx: rrf_score} отсортированный."""
    scores = defaultdict(float)
    for rank, idx in enumerate(dense_indices, 1):
        scores[idx] += 1.0 / (k + rank)
    for rank, idx in enumerate(sparse_indices, 1):
        scores[idx] += 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def search(query_text, k_dense=K_DENSE, k_sparse=K_SPARSE, top_n=TOP_N):
    # Dense
    q_emb = embed_texts([query_text], prefix='query: ')
    _, dense_idx = index.search(q_emb.astype(np.float32), k_dense)
    dense_indices = [int(i) for i in dense_idx[0] if i != -1]

    # Sparse BM25
    bm25_scores  = bm25.get_scores(query_text.lower().split())
    sparse_indices = np.argsort(bm25_scores)[::-1][:k_sparse].tolist()

    # RRF
    fused = rrf_fusion(dense_indices, sparse_indices, k=RRF_K)

    # Агрегируем по видео: для каждого video_file берём лучший чанк
    seen_videos = {}
    results = []

    for chunk_idx, rrf_score in fused:
        chunk = chunks[chunk_idx]
        vid   = chunk['video_file']

        if vid not in seen_videos:
            seen_videos[vid] = True
            results.append({
                'video_file': vid,
                'start': chunk['start'],
                'end':   chunk['end'],
                'score': rrf_score,
            })

        if len(results) >= top_n:
            break

    return results

## 5. Формирование submission.csv

In [ ]:
test_df = pd.read_csv(TEST_CSV)
print(f'Запросов: {len(test_df)}')
print(test_df.head(3))

In [ ]:
rows = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Поиск'):
    qid     = row['query_id']
    query   = row['question']
    results = search(query, top_n=TOP_N)

    out = {'query_id': qid}
    for rank in range(1, TOP_N + 1):
        if rank <= len(results):
            r = results[rank - 1]
            out[f'video_file_{rank}'] = r['video_file']
            out[f'start_{rank}']      = r['start']
            out[f'end_{rank}']        = r['end']
        else:
            out[f'video_file_{rank}'] = ''
            out[f'start_{rank}']      = 0.0
            out[f'end_{rank}']        = 0.0
    rows.append(out)

# Правильный порядок колонок
cols = ['query_id']
for rank in range(1, TOP_N + 1):
    cols += [f'video_file_{rank}', f'start_{rank}', f'end_{rank}']

submission_df = pd.DataFrame(rows)[cols]
submission_df.to_csv('submission.csv', index=False)

print(f'Сохранено submission.csv — {len(submission_df)} строк')
print(submission_df.head(3))

In [ ]:
# Быстрая проверка формата
assert list(submission_df.columns) == cols, 'Неправильные колонки!'
assert len(submission_df) == len(test_df), 'Кол-во строк не совпадает!'
assert submission_df['video_file_1'].str.match(r'^video_[a-f0-9]+$').mean() > 0.9, 'Подозрительные video_file!'
print('Проверка пройдена!')